# Notebook 1 Explained: Code Snippet Guide

This notebook explains every code snippet from Notebook 1 in simple terms.

For each snippet, you get:
- Why it is used
- How it works
- How to change it if needed

## Snippet 1 (Notebook 1 - Cell 3): Package Installation Check

### Why this is used
It makes sure required libraries are available before running analysis. This avoids import errors later.

### How it works
- Loops through a package list.
- Tries importing each package with __import__.
- If import fails, installs using pip through subprocess and sys.executable.

### How to change it
- Add/remove package names in the packages list.
- Remove this cell if your environment is already managed with requirements.txt or conda.
- If installation is slow, install once in terminal and keep this cell as imports only.

In [ ]:
# Snippet 1: Package installation check
import subprocess
import sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn']
for package in packages:
    try:
        __import__(package)
        print(f"{package} already installed")
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f"{package} installed")

## Snippet 2 (Notebook 1 - Cell 4): Library Imports and Global Settings

### Why this is used
It loads all tools used across the notebook and configures reproducibility and display behavior.

### How it works
- Imports pandas, numpy, matplotlib, seaborn, sklearn tools, and warnings.
- warnings.filterwarnings('ignore') hides warning messages.
- np.random.seed(42) makes random operations reproducible.
- pandas and matplotlib settings improve table and chart readability.

### How to change it
- Remove unused imports to keep code clean.
- Change seed value (42) if you want a different reproducible run.
- Adjust figure size or display max rows/columns depending on your screen/report format.
- Keep warnings visible during debugging by removing filterwarnings line.

In [ ]:
# Snippet 2: Imports and global settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Snippet 3 (Notebook 1 - Cell 5): Load Dataset and Initial Overview

### Why this is used
It loads your CSV file and gives a quick health check of the dataset.

### How it works
- Reads loan_approval_data.csv into dataframe df.
- Prints dataset shape, preview rows, data types, missing values, and summary statistics.

### How to change it
- Change file name/path in pd.read_csv if dataset location changes.
- Use encoding argument if CSV has special characters.
- Replace df.head() with df.sample(5) to view random rows.

In [ ]:
# Snippet 3: Load dataset and initial overview
df = pd.read_csv('loan_approval_data.csv')
print(f"Shape: {df.shape}")
print(df.head())
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())

## Snippet 4 (Notebook 1 - Cell 8): Descriptive Statistics of Retained Variables

### Why this is used
It removes non-modeling columns and summarizes the retained input variables.

### How it works
- Drops id and max_allowed_loan to form retained_vars.
- Uses describe() to compute count, mean, std, min/max, and quartiles for numeric columns.

### How to change it
- Update drop list based on feature selection decisions.
- Use describe(include='all') if you also want categorical summary metrics.
- Keep max_allowed_loan if you later decide to model it as a target in regression tasks.

In [ ]:
# Snippet 4: Descriptive statistics on retained variables
retained_vars = df.drop(['id', 'max_allowed_loan'], axis=1)
print(retained_vars.describe())

## Snippet 5 (Notebook 1 - Cell 9): Variable Scale Type Classification

### Why this is used
It documents each variable scale type, which helps justify preprocessing and model choices.

### How it works
- Iterates through retained_vars columns.
- Uses dtype checks: object -> categorical, numeric -> interval/continuous, and target -> binary nominal.
- Stores results in scale_types dictionary and converts to dataframe.

### How to change it
- Add explicit rules for ordinal variables if any exist.
- If a numeric code is actually categorical, cast it to string first before classification.
- Export the table to CSV if you want it in the report appendix.

In [ ]:
# Snippet 5: Scale type classification
scale_types = {}
for col in retained_vars.columns:
    if retained_vars[col].dtype == 'object':
        scale_types[col] = 'Nominal/Categorical'
    elif retained_vars[col].dtype in ['int64', 'float64']:
        scale_types[col] = 'Nominal (Binary)' if col == 'loan_approval_status' else 'Continuous/Interval'

scale_df = pd.DataFrame(list(scale_types.items()), columns=['Variable', 'Scale Type'])
print(scale_df.to_string(index=False))

## Snippet 6 (Notebook 1 - Cell 10): Target Distribution Analysis and Visualization

### Why this is used
It checks class balance in the target variable and visualizes approval vs rejection.

### How it works
- Computes value counts for loan_approval_status.
- Calculates approval percentage.
- Draws bar chart (absolute counts) and pie chart (proportions).

### How to change it
- Replace pie chart with countplot if you prefer clearer comparison.
- Add percentage labels on bar chart for report-ready visuals.
- If target labels change, update axis tick labels accordingly.

In [ ]:
# Snippet 6: Target distribution and visualization
target_distribution = df['loan_approval_status'].value_counts()
print(target_distribution)
print(f"Approval rate: {target_distribution.get(1, 0) / len(df) * 100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['loan_approval_status'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_xticklabels(['Rejected (0)', 'Approved (1)'], rotation=0)
df['loan_approval_status'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## Snippet 7 (Notebook 1 - Cell 12): Data Quality Assessment Setup

### Why this is used
It creates a safe working copy and identifies key column groups needed for cleaning.

### How it works
- Copies original df to df_processed so source data remains unchanged.
- Computes missing values summary.
- Detects categorical and numerical columns with select_dtypes.

### How to change it
- Exclude target from numerical_cols if you do not want to include it in outlier checks.
- Add datetime detection if future datasets contain date columns.
- Keep an untouched backup with df_raw = df.copy() when doing multiple preprocessing experiments.

In [ ]:
# Snippet 7: Data quality assessment setup
df_processed = df.copy()
missing = df_processed.isnull().sum()
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()

print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found')
print('Categorical:', categorical_cols)
print('Numerical:', numerical_cols)

## Snippet 8 (Notebook 1 - Cell 13): Issue Detection and Justification Table

### Why this is used
It systematically lists quality problems and documents justified fixes for coursework reporting.

### How it works
- Builds issues_found as a list of dictionaries.
- Checks missing values per column.
- Checks duplicate rows globally.
- Detects outliers per numerical feature using IQR rule.
- Adds class imbalance note for target.
- Converts list to dataframe and prints an audit table.

### How to change it
- Use z-score or percentile rules instead of IQR for outlier detection if needed.
- Add severity levels (Low/Medium/High) in each issue record.
- Save issues_df to CSV for reproducible evidence in report submission.

In [ ]:
# Snippet 8: Issue detection table
issues_found = []

for col in df_processed.columns:
    null_count = df_processed[col].isnull().sum()
    if null_count > 0:
        issues_found.append({
            'Variable': col,
            'Issue': f'Missing values ({null_count})',
            'Proposed Fix': 'Mean/Mode imputation',
            'Justification': 'Keeps dataset usable and consistent'
        })

if df_processed.duplicated().sum() > 0:
    issues_found.append({
        'Variable': 'All',
        'Issue': f"Duplicate rows ({df_processed.duplicated().sum()})",
        'Proposed Fix': 'Drop duplicates',
        'Justification': 'Avoids bias from repeated records'
    })

issues_df = pd.DataFrame(issues_found) if issues_found else pd.DataFrame()
print(issues_df.to_string(index=False) if not issues_df.empty else 'No critical issues found')

## Snippet 9 (Notebook 1 - Cell 14): Apply Data Cleaning Fixes

### Why this is used
It performs actual cleaning operations and prints before/after metrics to prove impact.

### How it works
- Prints initial shape, total missing values, and duplicates.
- For categorical columns: fills NaN with mode, fallback Unknown if mode unavailable.
- For numerical columns: fills NaN with mean.
- Removes duplicate rows using drop_duplicates.
- Prints final quality checks after transformations.

### How to change it
- Replace mean with median for skewed numeric distributions.
- Use model-based imputation (for example KNNImputer) for advanced handling.
- Consider not dropping duplicates if repeated records are meaningful business events.

In [ ]:
# Snippet 9: Apply cleaning fixes
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        fill_value = df_processed[col].mode()[0] if len(df_processed[col].mode()) > 0 else 'Unknown'
        df_processed[col] = df_processed[col].fillna(fill_value)

for col in numerical_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col] = df_processed[col].fillna(df_processed[col].mean())

df_processed.drop_duplicates(inplace=True)
print('Missing values after cleaning:', df_processed.isnull().sum().sum())
print('Duplicates after cleaning:', df_processed.duplicated().sum())

## Snippet 10 (Notebook 1 - Cell 15): Cleaned Dataset Summary

### Why this is used
It gives a final sanity check of cleaned data before saving and modeling.

### How it works
- Prints final shape and quality metrics.
- Shows first rows of df_processed for quick visual verification.

### How to change it
- Add df_processed.describe() for full numeric summary after cleaning.
- Add assertion checks (for example missing values must be zero).
- Show random sample instead of head for broader inspection.

In [ ]:
# Snippet 10: Final cleaned summary
print(f"Dataset shape: {df_processed.shape}")
print(f"Missing values: {df_processed.isnull().sum().sum()}")
print(f"Duplicate rows: {df_processed.duplicated().sum()}")
print(df_processed.head())

## Snippet 11 (Notebook 1 - Cell 16): Save Artifacts for Notebook 2

### Why this is used
It persists prepared data and encoding metadata so Notebook 2 can reuse exactly the same preprocessing outputs.

### How it works
- Imports joblib for object serialization.
- Saves df_processed to data_processed.joblib.
- Label-encodes categorical columns (except id), stores encoders in dictionary.
- Saves encoder dictionary to label_encoders.joblib.

### How to change it
- If you want to keep original columns only, remove creation of _encoded columns.
- Save outputs to a dedicated folder like artifacts/ for cleaner project structure.
- Add versioned filenames (for example data_processed_v2.joblib) when testing multiple preprocessing pipelines.

In [ ]:
# Snippet 11: Save artifacts for Notebook 2
import joblib
from sklearn.preprocessing import LabelEncoder

joblib.dump(df_processed, 'data_processed.joblib')

label_encoders = {}
for col in categorical_cols:
    if col != 'id':
        le = LabelEncoder()
        df_processed[col + '_encoded'] = le.fit_transform(df_processed[col])
        label_encoders[col] = le

joblib.dump(label_encoders, 'label_encoders.joblib')
print('Saved data_processed.joblib and label_encoders.joblib')

## Recommended Improvements for Future Changes

1. Move package installation out of notebook and use a requirements file.
2. Add a config section (file paths, target column, dropped columns) at top of notebook.
3. Save cleaned dataset as CSV and joblib for easier sharing and inspection.
4. Add lightweight validation tests before saving artifacts.